# Precision, Recall & Threshold Tuning Lab

Accuracy treats all classification errors equally, but in real engineering applications, the costs of **False Positives (false alarms)** and **False Negatives (missed cases)** are vastly asymmetric. This lab explores the mathematical definitions of Precision and Recall, demonstrates the mechanical tradeoff governed by the decision threshold $\tau$, and calculates Specificity and True Positive Rates across varying problem domains.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, confusion_matrix, precision_recall_curve
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Spam Filtering: Prioritizing Precision

In email filtering, a false positive (flagging a critical email as spam) is far more disruptive than letting a spam email slip into the inbox.

In [ ]:
# 20 emails: 1 = Spam, 0 = Not Spam
y_true = np.array([0]*10 + [1]*10)
y_pred = np.array([0, 0, 0, 1, 0, 1, 0, 0, 0, 0,  # 2 False Positives
                   1, 1, 0, 1, 1, 1, 1, 0, 1, 1])  # 2 False Negatives

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)

print(f"Spam Filter Confusion Matrix:")
print(f"  TN={tn}, FP={fp} (False Alarms)")
print(f"  FN={fn} (Missed Spam), TP={tp}")
print(f"\nPrecision: {prec:.1%} (When filter says spam, it is right {prec:.1%} of the time)")
print(f"Recall:    {rec:.1%} (Filter catches {rec:.1%} of all true spam)")

## 2. Disease Screening: Prioritizing Recall

In medical oncology screening, a false negative (failing to diagnose an aggressive tumor) is potentially fatal, so we tune for maximum recall even at the cost of false alarms.

In [ ]:
# 100 patients: 85 Healthy (0), 15 Sick (1)
y_med = np.concatenate([np.zeros(85, dtype=int), np.ones(15, dtype=int)])

# High-recall screening model: catches 13/15 sick patients, flags 8 healthy patients
y_med_pred = y_med.copy()
y_med_pred[85:87] = 0  # 2 False Negatives
y_med_pred[:8] = 1     # 8 False Positives

cm_med = confusion_matrix(y_med, y_med_pred)
tn_m, fp_m, fn_m, tp_m = cm_med.ravel()

prec_m = tp_m / (tp_m + fp_m)
rec_m = tp_m / (tp_m + fn_m)
spec_m = tn_m / (tn_m + fp_m)

print(f"Medical Screening Results:")
print(f"  Precision:   {prec_m:.1%} (Lower precision due to false alarms)")
print(f"  Recall:      {rec_m:.1%} (High sensitivity: caught {tp_m}/15 cases)")
print(f"  Specificity: {spec_m:.1%} (Correctly cleared {tn_m}/85 healthy patients)")

## 3. The Precision-Recall Threshold Tradeoff

Vary the classification threshold $\tau \in [0.2, 0.8]$ over continuous predicted probability scores to demonstrate how raising the threshold drives Precision up and Recall down.

In [ ]:
# Generate continuous scores
n_neg, n_pos = 60, 40
y_scores_true = np.concatenate([np.zeros(n_neg, dtype=int), np.ones(n_pos, dtype=int)])
scores = np.concatenate([
    np.random.beta(2, 5, n_neg),  # Negatives clustered around 0.28
    np.random.beta(5, 2, n_pos)   # Positives clustered around 0.71
])

thresholds = [0.25, 0.40, 0.50, 0.60, 0.75]
print(f"{'Threshold':<12} {'TP':<5} {'FP':<5} {'FN':<5} {'Precision':<12} {'Recall':<12} {'F1-Score'}")
print("-" * 68)

for thresh in thresholds:
    y_t = (scores >= thresh).astype(int)
    p = precision_score(y_scores_true, y_t, zero_division=0)
    r = recall_score(y_scores_true, y_t, zero_division=0)
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    tp_t = np.sum((y_scores_true == 1) & (y_t == 1))
    fp_t = np.sum((y_scores_true == 0) & (y_t == 1))
    fn_t = np.sum((y_scores_true == 1) & (y_t == 0))
    print(f"{thresh:<12.2f} {tp_t:<5} {fp_t:<5} {fn_t:<5} {p:<12.1%} {r:<12.1%} {f1:.3f}")